In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [11]:
from langchain_core.documents import Document

In [12]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [13]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [14]:
type(sample_doc)

langchain_core.documents.base.Document

In [15]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data1/Python.txt", encoding="utf-8")

In [16]:
document = loader.load()

In [17]:
document

[Document(metadata={'source': 'data1/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and m

In [20]:
# PDF data
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("data1/research2.pdf")

document = pdf_loader.load()
document

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-03-28T00:54:45+00:00', 'author': '', 'keywords': '', 'moddate': '2024-03-28T00:54:45+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data1/research2.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}, page_content='1\nRetrieval-Augmented Generation for Large\nLanguage Models: A Survey\nYunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng\nWangc, and Haofen Wang a,c\naShanghai Research Institute for Intelligent Autonomous Systems, Tongji University\nbShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University\ncCollege of Design and Innovation, Tongji University\nAbstract—Large Language Models (LLMs) showcase impres-\nsive capabilities but encounter challenges like hallu

In [19]:
# PDF data
from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_loader = PyMuPDFLoader("data1/research2.pdf")

document = pdf_loader.load()
document

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-03-28T00:54:45+00:00', 'source': 'data1/research2.pdf', 'file_path': 'data1/research2.pdf', 'total_pages': 21, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-03-28T00:54:45+00:00', 'trapped': '', 'modDate': 'D:20240328005445Z', 'creationDate': 'D:20240328005445Z', 'page': 0}, page_content='1\nRetrieval-Augmented Generation for Large\nLanguage Models: A Survey\nYunfan Gaoa, Yun Xiongb, Xinyu Gaob, Kangxiang Jiab, Jinliu Panb, Yuxi Bic, Yi Daia, Jiawei Suna, Meng\nWangc, and Haofen Wang a,c\naShanghai Research Institute for Intelligent Autonomous Systems, Tongji University\nbShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University\ncCollege of Design and Innovation, Tongji University\nAbstract—Large Language Models (LLMs) showcase impres-\nsive capabilities but encounter challenges like hallucination,\noutd

## Ingestion Pipeline

In [21]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [22]:
def load_all_pdfs():
    folder_path = "data1/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [23]:
all_pdf_documents = load_all_pdfs()

total pdfs: 1
total pages: 21


In [24]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [25]:
# chunks
!pip install langchain_text_splitters

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [27]:
chunks = split_docs(all_pdf_documents)

In [28]:
len(chunks)

244

### Embedding

In [29]:
from sentence_transformers import SentenceTransformer

In [30]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [31]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


### Vector Store

In [32]:
import chromadb
import uuid

In [33]:
class VectorStoreManager:
    def __init__(self, persist_directory="data1/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [34]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 732


In [35]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
total documents added in vector store= 244
docs in collection: 976


# Retrieval Pipeline

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [38]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [39]:
rag_retriever.retrieve("What is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_e0a8d23a-5a0e-4a20-9c4e-9a1b211d6753',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'trapped': '/False',
   'content_length': 288,
   'doc_index': 11,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'subject': '',
   'author': '',
   'total_pages': 21,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page': 0,
   'keywords': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'creator': 'LaTeX with hyperref',
   'producer': 'pdfTeX-1.40.25',
   'page_label': '1',
   'source': 'data1/pdfs\\research2.pdf',
   'title': ''},
  'distance': 0.4226756691932678,
  'similarity_score': 0.5773243308067322,
  'rank': 1},
 {'id': 'doc_9a70e959

# Integrate with LLMs

## OpenAI - GPT

In [61]:
API_KEY_OPENAI = ""

In [62]:
!pip install langchain-openai

In [63]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [64]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

In [65]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents
we found no relevant context for the given query


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [66]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me look at the provided context to figure out the answer. The context mentions "RAG methods" and talks about a survey that reviews state-of-the-art RAG methods, including paradigms like naive RAG. The arXiv reference is from March 2024, so it's a recent paper.

First, I need to recall what RAG stands for. From what I remember, RAG is Retrieval-Augmented Generation. It's a method that combines retrieval of information from a database with a generative model to enhance the output. The context here supports that since it's discussing different paradigms of RAG, like naive RAG, which probably refers to the basic approach where the model retrieves documents and then generates a response based on them.

The user might be someone interested in NLP or machine learning, looking for a concise definition. They might not know the full form or the underlying concepts. The answer should explain RAG in simple terms, mention its components (retrieva

### Groq

In [42]:
API_Key_GROQ = ""

In [43]:
!pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   -------------------- ------------------- 1/2 [langchain-groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [49]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [50]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [51]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [52]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me look at the provided context to figure out the answer. The context mentions "RAG methods" and talks about a survey that reviews state-of-the-art RAG methods, including paradigms like naive RAG. The arXiv reference is from March 2024, so it's a recent paper.

First, I need to recall what RAG stands for. From what I remember, RAG is Retrieval-Augmented Generation. It's a method that combines retrieval of information from a database with a generative model to enhance the output. The context here supports that since it's discussing different paradigms of RAG, like naive RAG, which probably refers to the basic approach where the model retrieves documents and then generates a response based on them.

The user might be someone interested in NLP or machine learning, looking for a concise definition. They might not know the full form or the underlying concepts. The answer should explain RAG in simple terms, mention its components (retrieva

# CLAUDE

In [53]:
# Claude
!pip install langchain-anthropic


   ---------------------------------------- 0/3 [docstring-parser]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- -------------------------- 1/3 [anthropic]
   ------------- ---------------

In [57]:
API_KEY_CLAUDE = ""

In [58]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    api_key=API_KEY_CLAUDE,
    model="claude-3-sonnet-20240229",
    temperature=0.1,
    max_tokens=1024
)

In [59]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [60]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CZiJSo9mvAuoUjnnCEQkd'}

In [ ]:
print(answer)